In [1]:
import scipy.io
import networkx as nx 
import bct 
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
import nilearn
from nilearn import datasets, plotting, surface
import random
import seaborn as sns

# Preparation of dataset for use
number_subjects = 9

# definition of directory with dataset
dir_fmri_desikan = '/strombolihome/fribeiro/Dataset/source_reconstructed_FC/fmri_connect_desikan/'
dir_fmri_destrieux = '/strombolihome/fribeiro/Dataset/source_reconstructed_FC/fmri_connect_destrieux/'

dir_eeg_desikan = '/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_desikan/'
dir_eeg_destrieux = '/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_destrieux/'

# importing labels of atlas
mat_desikan = scipy.io.loadmat('/strombolihome/fribeiro/Dataset/source_reconstructed_FC/label_dsk.mat', squeeze_me=True)
labels_desikan = mat_desikan['label_dsk']
labels_desikan[67] = 'rINS' # small correction

mat_destrieux = scipy.io.loadmat('/strombolihome/fribeiro/Dataset/source_reconstructed_FC/label_dstrx.mat', squeeze_me=True)
labels_destrieux = mat_destrieux['label_dstrx']
labels_destrieux = labels_destrieux[12:160]
labels_destrieux[147] = 'rS_temporal_transverse' # small correction


In [2]:
# Script for creating graphs from fMRI and EEG connectivity data (coverting MATLAB matrices)

#1.Creation of average Graph 
def createAvGraph(file,data_type):
    
    # Load Matlab file with connectivity matrix for each time point - 3D matrix
    mat = scipy.io.loadmat(file)

    if data_type == 'fmri':
        conn_matrix = mat['connFMRI'].transpose() #connectivity fMRI matrix with numpy format
    elif data_type == 'eeg': #for now only for broad band TODO others !
        conn_matrix = mat['connEEGbroad'].transpose() #connectivity EEG matrix with numpy format
    elif data_type == 'eeg_alpha':
        conn_matrix = mat['connEEGalpha'].transpose()
    elif data_type == 'eeg_beta':
        conn_matrix = mat['connEEGbeta'].transpose()
    elif data_type == 'eeg_delta':
        conn_matrix = mat['connEEGdelta'].transpose()
    elif data_type == 'eeg_gamma':
        conn_matrix = mat['connEEGgamma'].transpose()
    elif data_type == 'eeg_theta':
        conn_matrix = mat['connEEGtheta'].transpose()
        
    t_points = conn_matrix.shape[0] # number of layers of multilayer matrix
    num_areas = conn_matrix.shape[1] #number of nodes of the graph
    
    # Get average connectivity matrix
    av_conn = np.zeros((num_areas,num_areas))
       
    for i in range(t_points):
        
        av_conn = av_conn + conn_matrix[i]
            
    av_conn = av_conn/t_points 
    
    G = nx.from_numpy_matrix(av_conn)
    
    return G

#2. Creation of array of graphs (equivalent to layers)
def createArrayGraph(file,data_type):
    
    # Load Matlab file with connectivity matrix for each time point - 3D matrix
    mat = scipy.io.loadmat(file)
    
    if data_type == 'fmri':
        conn_matrix = mat['connFMRI'].transpose() #connectivity fMRI matrix with numpy format
    elif data_type == 'eeg': #for now only for broad band TODO others !
        conn_matrix = mat['connEEGbroad'].transpose() #connectivity EEG matrix with numpy format
    elif data_type == 'eeg_alpha':
        conn_matrix = mat['connEEGalpha'].transpose()
    elif data_type == 'eeg_beta':
        conn_matrix = mat['connEEGbeta'].transpose()
    elif data_type == 'eeg_delta':
        conn_matrix = mat['connEEGdelta'].transpose()
    elif data_type == 'eeg_gamma':
        conn_matrix = mat['connEEGgamma'].transpose()
    elif data_type == 'eeg_theta':
        conn_matrix = mat['connEEGtheta'].transpose()

    t_points = conn_matrix.shape[0] # number of layers of multilayer matrix
    num_areas = conn_matrix.shape[1] #number of nodes of the graph
    
    array_graphs = np.empty(t_points, dtype=object) 
    
    for i in range(t_points):
        array_graphs[i] = nx.from_numpy_matrix(conn_matrix[i])
        
    return array_graphs

#concatenating all subjects
def createArrayGraphAllSubjects(file,type_data):
    
    array_graphs_all = []
    
    for f in range(0,len(file)):
        array_graphs = createArrayGraph(file[f], type_data)
        if f == 0:
            array_graphs_all = array_graphs
        else:
            array_graphs_all = np.concatenate([array_graphs_all, array_graphs])
    
    return array_graphs_all

#3. Threshold graph with given proportion
def thresholdGraph(G, threshold, type_data, type_threshold):
    
    if(type_data == 'fmri'):
        # get absolute value connectivity matrix
        conn_matrix = abs(nx.to_numpy_array(G))
    else:
        conn_matrix = nx.to_numpy_array(G)
     
    if (type_threshold == 'abs'):
        # to threshold graph keeping edges with weight equal or above certain value
        conn_new = bct.utils.threshold_absolute(conn_matrix, threshold, True)
    else:
        # to threshold graph keeping the top % of the edges 
        conn_new = bct.utils.threshold_proportional(conn_matrix, threshold, True)
        #add all weights equal to the minimal value kept
        min_val = np.min(conn_new[np.nonzero(conn_new)])
        index = np.transpose(np.where(conn_matrix == min_val))
        idx_i, idx_j = zip(*index)
        conn_new[idx_i, idx_j] = conn_matrix[idx_i, idx_j]
    
    #for the fMRI data we need to recover non-absolute values as Phase Coherence is between -1 and 1
    if(type_data == 'fmri'):
        
        ind_keep = np.transpose(np.nonzero(conn_new))
        conn_new = np.zeros((G.number_of_nodes(), G.number_of_nodes()))
        min_positive = 1 #to store minimum positive value kept of phase coherence
        max_negative = -1 #to store maximum negative value kept of phase coherence
        
        idx_i, idx_j = zip(*ind_keep)

        conn_new[idx_i, idx_j] = nx.to_numpy_array(G)[idx_i, idx_j]
        
        # remove self loops
        np.fill_diagonal(conn_new, 0)
        
        #In case we want the minimum value of connectivity kept
        
        #for ind in ind_keep:
        #    #print(nx.to_numpy_array(G)[ind[0],ind[1]])
        #    conn_new[ind[0],ind[1]] = nx.to_numpy_array(G)[ind[0],ind[1]]
        #    #print(conn_new[ind[0],ind[1]])
        #    if nx.to_numpy_array(G)[ind[0],ind[1]] > 0 and nx.to_numpy_array(G)[ind[0],ind[1]] < min_positive:
        #        min_positive = nx.to_numpy_array(G)[ind[0],ind[1]]
        #    elif nx.to_numpy_array(G)[ind[0],ind[1]] < 0 and nx.to_numpy_array(G)[ind[0],ind[1]] > max_negative:
        #        max_negative = nx.to_numpy_array(G)[ind[0],ind[1]]
        
        G_new = nx.from_numpy_matrix(conn_new)
        
        return [G_new, conn_new, min_positive, max_negative]
    
    else:
        min_coh = np.amin(conn_new) #to store minimum value kept of imaginary part of coherency
        G_new = nx.from_numpy_matrix(conn_new)
        
        return [G_new, conn_new, min_coh]  

#4. Get the components of the graph after thresholding
def getThresholdComponents(G,threshold,data, type_threshold):
    
    if data == 'fmri':
        G_new, conn_matrix, min_pos, max_neg = thresholdGraph(G, threshold, data, type_threshold)
    else:
        G_new, conn_matrix, min_coh = thresholdGraph(G, threshold, data, type_threshold)
        
    graph_components = sorted(nx.connected_components(G_new), key=len, reverse=True)
    
    return graph_components


In [18]:
#5. Generate txt file of the network

def getGraphTxtFile(G):
    
    f = open("/strombolihome/fribeiro/Thesis_project/Code/test.txt", "a")
    for edge in list(G.edges(data=True)):
        f.write(str(edge[1]+1) + ' ' + str(edge[0]+1) + ' ' + str(int(G[edge[0]][edge[1]]["weight"])) + '\n')
    f.close()
    
    return f
    

G_array = createArrayGraph(dir_fmri_desikan + 'subj06-7T/conn_desi_phase_coh_time_fmri.mat', 'fmri')
G, conn_matrix, min_pos, max_neg = thresholdGraph(G_array[3], 0.11, 'fmri', 'percentage')
conn_matrix = bct.utils.binarize(conn_matrix)
G = nx.from_numpy_matrix(conn_matrix)

f = open("/strombolihome/fribeiro/Thesis_project/Code/test.txt", "a")
for edge in list(G.edges(data=True)):
    f.write(str(edge[1]+1) + ' ' + str(edge[0]+1) + ' ' + str(int(G[edge[0]][edge[1]]["weight"])) + '\n')
f.close()

#with open('/strombolihome/fribeiro/Thesis_project/Code/test.txt', 'a') as the_file:
#    for edge in list(G.edges(data=True)):
#        the_file.write()
#    the_file.write('Hello\n')

for edge in list(G.edges(data=True)):
    print(edge)
    print(edge[0])
    print(G[edge[0]][edge[1]]["weight"])
#print(list(G.edges(data=True)))

(0, 1, {'weight': 1.0})
0
1.0
(0, 4, {'weight': 1.0})
0
1.0
(0, 8, {'weight': 1.0})
0
1.0
(0, 10, {'weight': 1.0})
0
1.0
(0, 14, {'weight': 1.0})
0
1.0
(0, 25, {'weight': 1.0})
0
1.0
(0, 26, {'weight': 1.0})
0
1.0
(0, 31, {'weight': 1.0})
0
1.0
(0, 33, {'weight': 1.0})
0
1.0
(0, 42, {'weight': 1.0})
0
1.0
(0, 46, {'weight': 1.0})
0
1.0
(0, 58, {'weight': 1.0})
0
1.0
(0, 61, {'weight': 1.0})
0
1.0
(0, 65, {'weight': 1.0})
0
1.0
(0, 66, {'weight': 1.0})
0
1.0
(1, 4, {'weight': 1.0})
1
1.0
(1, 9, {'weight': 1.0})
1
1.0
(1, 11, {'weight': 1.0})
1
1.0
(1, 12, {'weight': 1.0})
1
1.0
(1, 26, {'weight': 1.0})
1
1.0
(1, 30, {'weight': 1.0})
1
1.0
(1, 31, {'weight': 1.0})
1
1.0
(1, 42, {'weight': 1.0})
1
1.0
(1, 46, {'weight': 1.0})
1
1.0
(1, 58, {'weight': 1.0})
1
1.0
(1, 61, {'weight': 1.0})
1
1.0
(1, 64, {'weight': 1.0})
1
1.0
(1, 65, {'weight': 1.0})
1
1.0
(2, 29, {'weight': 1.0})
2
1.0
(2, 48, {'weight': 1.0})
2
1.0
(3, 15, {'weight': 1.0})
3
1.0
(3, 23, {'weight': 1.0})
3
1.0
(3, 53, {'wei

In [220]:
G = createAvGraph(dir_fmri_desikan + 'subj01-7T/conn_desi_phase_coh_time_fmri.mat', 'fmri')
G, conn_matrix, min_pos, max_neg = thresholdGraph(G, 0.11, 'fmri', 'percentage')
conn_matrix = bct.utils.binarize(conn_matrix)
G = nx.from_numpy_matrix(conn_matrix)

f = open("/strombolihome/fribeiro/Thesis_project/Code/graph.txt", "a")
for edge in list(G.edges(data=True)):
    f.write(str(edge[1]+1) + ' ' + str(edge[0]+1) + ' ' + str(int(G[edge[0]][edge[1]]["weight"])) + '\n')
f.close()

#open("/strombolihome/fribeiro/Thesis_project/Code/test.txt", 'w').close()

In [3]:
import re
import pandas as pd

def arrangeMotifResults(file, mode):
    
    f = open(file, 'r')
    lines = f.readlines() 
    words = []
    i = 1
    for line in lines: 
        line = format(line.strip())
        #print(line)
        # when the results start
        if i >= 29:
            #to remove additional characters
            out = re.split(' |, |\*|\n',line)
            out = [o for o in out if o != '|'] 
            out = [o for o in out if o != '' ]
            out = [o for o in out if o != '+/-']
            if len(out) > 0:
                words.append(out[:])

        i += 1
    
    #print(words)
    f.close()
    
    #if studying motifs with size 3 - 2 types
    if mode == '3':
        
        df = pd.DataFrame(columns = ['Subgraph Code', 'Frequency', 'Z-score', 'Random_av', 'Random_dev'], 
                              index = ['011#100#100', '011#101#110'])
        code = ''
        for j in range(0,len(words)):
            j += 1
            #print("make codes")
            if j%3 != 0:
                code += str(words[j-1][0]) + "#"
            elif j%3 == 0:
                code += str(words[j-1][0])
            #print(code)
            if code == '011#100#100':
                #print("here")
                df.loc[code] = ['1', float(words[j-1][1]), float(words[j-1][2]), float(words[j-1][3]), float(words[j-1][4])]  
                code = ''
            elif code == '011#101#110':
                #print("here 2")
                df.loc[code] = ['2', float(words[j-1][1]), float(words[j-1][2]), float(words[j-1][3]), float(words[j-1][4])]  
                code = ''
        return df
    #or if studying motifs with size 4 - 6 types      
    elif mode == '4':
        df = pd.DataFrame(columns = ['Subgraph Code', 'Frequency', 'Z-score', 'Random_av', 'Random_dev'], 
                              index = ['0110#1001#1000#0100', '0111#1010#1100#1000', '0111#1000#1000#1000', 
                                       '0111#1011#1100#1100', '0111#1011#1101#1110', '0110#1001#1001#0110'])
        code = ''
        for j in range(0,len(words)):
            j += 1
            
            if j%4 != 0:
                code += str(words[j-1][0]) + "#"
            elif j%4 == 0:
                code += str(words[j-1][0])
            if code == '0110#1001#1000#0100':
                #print("here")
                df.loc[code] = ['1', float(words[j-1][1]), float(words[j-1][2]), float(words[j-1][3]), float(words[j-1][4])]  
                code = ''
            elif code == '0111#1010#1100#1000':
                df.loc[code] = ['2', float(words[j-1][1]), float(words[j-1][2]), float(words[j-1][3]), float(words[j-1][4])]  
                code = ''
            elif code == '0111#1000#1000#1000':
                df.loc[code] = ['3', float(words[j-1][1]), float(words[j-1][2]), float(words[j-1][3]), float(words[j-1][4])]  
                code = ''
            elif code == '0111#1011#1100#1100':
                df.loc[code] = ['4', float(words[j-1][1]), float(words[j-1][2]), float(words[j-1][3]), float(words[j-1][4])]    
                code = ''
            elif code == '0111#1011#1101#1110':
                df.loc[code] = ['5', float(words[j-1][1]), float(words[j-1][2]), float(words[j-1][3]), float(words[j-1][4])]   
                code = ''
            elif code == '0110#1001#1001#0110':
                df.loc[code] = ['6', float(words[j-1][1]), float(words[j-1][2]), float(words[j-1][3]), float(words[j-1][4])]    
                code = ''
        return df
    #or if studying motifs with size 5 - 21 types        
    elif mode == '5':
        df = pd.DataFrame(columns = ['Subgraph Code', 'Frequency', 'Z-score', 'Random_av', 'Random_dev'], 
                              index = ['01100#10010#10001#01000#00100', '01110#10001#10000#10000#01000', '01110#10100#11000#10001#00010',
                                      '01110#10101#11000#10000#01000', '01110#10110#11001#11000#00100', '01111#10100#11000#10000#10000',
                                       '01111#10110#11000#11000#10000', '01111#10110#11010#11100#10000', '01111#10110#11001#11000#10100',
                                      '01100#10011#10010#01100#01000', '01111#10111#11010#11100#11000', '01111#10100#11000#10001#10010',
                                      '01111#10000#10000#10000#10000', '01101#10011#10010#01100#11000', '01111#10111#11011#11100#11100',
                                      '01111#10111#11000#11000#11000', '01100#10010#10001#01001#00110', '01110#10110#11001#11001#00110',
                                      '01111#10110#11001#11001#10110', '01111#10111#11011#11101#11110', '01100#10011#10011#01100#01100'])
        code = ''
        for j in range(0,len(words)):
            j += 1
            
            if j%5 != 0:
                code += str(words[j-1][0]) + "#"
            elif j%5 == 0:
                code += str(words[j-1][0])
            if code == '01100#10010#10001#01000#00100':
                #print("here")
                df.loc[code] = ['1', float(words[j-1][1]), float(words[j-1][2]), float(words[j-1][3]), float(words[j-1][4])]   
                code = ''
            elif code == '01110#10001#10000#10000#01000':
                df.loc[code] = ['2', float(words[j-1][1]), float(words[j-1][2]), float(words[j-1][3]), float(words[j-1][4])]  
                code = ''
            elif code == '01110#10100#11000#10001#00010':
                df.loc[code] = ['3', float(words[j-1][1]), float(words[j-1][2]), float(words[j-1][3]), float(words[j-1][4])]    
                code = ''
            elif code == '01110#10101#11000#10000#01000':
                df.loc[code] = ['4', float(words[j-1][1]), float(words[j-1][2]), float(words[j-1][3]), float(words[j-1][4])]   
                code = ''
            elif code == '01110#10110#11001#11000#00100':
                df.loc[code] = ['5', float(words[j-1][1]), float(words[j-1][2]), float(words[j-1][3]), float(words[j-1][4])]    
                code = ''
            elif code == '01111#10100#11000#10000#10000':
                df.loc[code] = ['6', float(words[j-1][1]), float(words[j-1][2]), float(words[j-1][3]), float(words[j-1][4])]    
                code = ''
            elif code == '01111#10110#11000#11000#10000':
                df.loc[code] = ['7', float(words[j-1][1]), float(words[j-1][2]), float(words[j-1][3]), float(words[j-1][4])]   
                code = ''
            elif code == '01111#10110#11010#11100#10000':
                df.loc[code] = ['8', float(words[j-1][1]), float(words[j-1][2]), float(words[j-1][3]), float(words[j-1][4])]   
                code = ''
            elif code == '01111#10110#11001#11000#10100':
                df.loc[code] = ['9', float(words[j-1][1]), float(words[j-1][2]), float(words[j-1][3]), float(words[j-1][4])]    
                code = ''
            elif code == '01100#10011#10010#01100#01000':
                df.loc[code] = ['10', float(words[j-1][1]), float(words[j-1][2]), float(words[j-1][3]), float(words[j-1][4])]    
                code = ''
            elif code == '01111#10111#11010#11100#11000':
                df.loc[code] = ['11', float(words[j-1][1]), float(words[j-1][2]), float(words[j-1][3]), float(words[j-1][4])]    
                code = ''
            elif code == '01111#10100#11000#10001#10010':
                df.loc[code] = ['12', float(words[j-1][1]), float(words[j-1][2]), float(words[j-1][3]), float(words[j-1][4])]    
                code = ''
            if code == '01111#10000#10000#10000#10000':
                df.loc[code] = ['13', float(words[j-1][1]), float(words[j-1][2]), float(words[j-1][3]), float(words[j-1][4])]   
                code = ''
            elif code == '01101#10011#10010#01100#11000':
                df.loc[code] = ['14', float(words[j-1][1]), float(words[j-1][2]), float(words[j-1][3]), float(words[j-1][4])]    
                code = ''
            elif code == '01111#10111#11011#11100#11100':
                df.loc[code] = ['15', float(words[j-1][1]), float(words[j-1][2]), float(words[j-1][3]), float(words[j-1][4])]   
                code = ''
            elif code == '01111#10111#11000#11000#11000':
                df.loc[code] = ['16', float(words[j-1][1]), float(words[j-1][2]), float(words[j-1][3]), float(words[j-1][4])]   
                code = ''
            elif code == '01100#10010#10001#01001#00110':
                df.loc[code] = ['17', float(words[j-1][1]), float(words[j-1][2]), float(words[j-1][3]), float(words[j-1][4])]   
                code = ''
            elif code == '01110#10110#11001#11001#00110':
                df.loc[code] = ['18', float(words[j-1][1]), float(words[j-1][2]), float(words[j-1][3]), float(words[j-1][4])]   
                code = ''
            elif code == '01111#10110#11001#11001#10110':
                df.loc[code] = ['19', float(words[j-1][1]), float(words[j-1][2]), float(words[j-1][3]), float(words[j-1][4])]   
                code = ''
            elif code == '01111#10111#11011#11101#11110':
                df.loc[code] = ['20', float(words[j-1][1]), float(words[j-1][2]), float(words[j-1][3]), float(words[j-1][4])]   
                code = ''
            elif code == '01100#10011#10011#01100#01100':
                df.loc[code] = ['21', float(words[j-1][1]), float(words[j-1][2]), float(words[j-1][3]), float(words[j-1][4])]    
                code = ''
        return df
        

In [4]:
import shlex, subprocess
import re
import pandas as pd
import time

#function to get all types of subgraphs, their frequency and z-score compared to a rewiring null model (100 networks)
# done with significant time points common between fMRI and EEG alpha
def motifEnumerationRewiring(G_array, threshold, delay, subject, type_data, atlas, mode):
    
    #set args for running c++ code and to store results
    if mode == '3':
        command_line = './gtrieScanner_src_01/gtrieScanner -s 3 -m gtrie ./gtrieScanner_src_01/gtries/undir3.gt -g graph.txt ' \
                        + '-u -r 100'
    elif mode == '4':
        command_line = './gtrieScanner_src_01/gtrieScanner -s 4 -m gtrie ./gtrieScanner_src_01/gtries/undir4.gt -g graph.txt ' \
                        + '-u -r 100'
    elif mode == '5':
        command_line = './gtrieScanner_src_01/gtrieScanner -s 5 -m gtrie ./gtrieScanner_src_01/gtries/undir5.gt -g graph.txt ' \
                        + '-u -r 100'
                
    significant_time_points_rewiring = np.load('/strombolihome/fribeiro/Thesis_project/Results/comparison_null_model/significant_time_points/subj0' + str(subject) + '/significant_time_points_fmri_delay_' + str(delay) + '_eeg_alpha_' + atlas + '.npy')
    significant_time_points = significant_time_points_rewiring.astype(int)
    significant_time_points = significant_time_points - 1
    print(significant_time_points)
    print(len(significant_time_points))
    
    for t in significant_time_points:
        print(t)
        
        if type_data == 'fmri':
            G, conn_matrix, min_pos, max_neg = thresholdGraph(G_array[t+delay], threshold, 'fmri', 'percentage')
        else:
            G, conn_matrix, min_coh = thresholdGraph(G_array[t], threshold, type_data, 'percentage')
        
        #conversion to binary undirected graph
        conn_matrix = bct.utils.binarize(conn_matrix)
        G = nx.from_numpy_matrix(conn_matrix)
        
        # to reset and write in graph representation txt
        #open("/strombolihome/fribeiro/Thesis_project/Code/graph.txt", 'w').close()
        #file.truncate(0)
        f = open("/strombolihome/fribeiro/Thesis_project/Code/graph.txt", "a")
        f.truncate(0)
        for edge in list(G.edges(data=True)):
            f.write(str(edge[1]+1) + ' ' + str(edge[0]+1) + ' ' + str(int(G[edge[0]][edge[1]]["weight"])) + '\n')
        f.close()
        
        # run c++ code
        args = shlex.split(command_line)
        p = subprocess.Popen(args)
        
        time.sleep(0.5)
        
        # store results in a dataframe for easy access and to clear out extra information
        df = arrangeMotifResults('/strombolihome/fribeiro/Thesis_project/Code/results.txt', mode)
        
        df.to_pickle('/strombolihome/fribeiro/Thesis_project/Results/motif_enumeration_null_model/subgraph_' + mode \
                        + '/subj0' + str(subject) + '/' + type_data + '/motifs_size_' + mode + '_' + type_data + '_' + str(threshold) + '_time_point_' + str(t) + '_' + atlas + '.pkl')


In [402]:
G_array = createArrayGraph(dir_fmri_desikan + 'subj01-7T/conn_desi_phase_coh_time_fmri.mat', 'fmri')
motifEnumerationRewiring(G_array, 0.11, 3, 1, 'fmri', 'dsk', '3')

[  0   1   2   3   4   5   6   8   9  11  13  15  17  19  20  21  22  26
  28  31  32  34  35  36  37  38  39  40  45  46  47  48  49  50  51  52
  54  55  56  57  58  59  61  62  63  64  65  66  67  68  69  70  71  72
  73  74  76  78  79  80  83  84  86  87  88  89  90  92  93  94  95  98
  99 100 102 103 104 105 106 107 109 111 113 118 119 120 122 124 126 131
 132 134 135 136 140 142 143 144 146 147 149 150 151 152 153 155 156 157
 159 160 164 165 167 168 169 170 174 176 177 178 179 180 181 182 183 184
 187 188 189 190 192 193 194 196 197 200 202 204 206 207 208 209 210 212
 213 214 216 218 219 220 221 223 224 225 227 229 230 231 235 237 239 240
 241 242 245 246 247 248 250 251 253 255 257 258 263 264 265 267 268 269
 270 272 274 275 276 277 279 280 281 282 283 285 286 287 291 292 293 294
 295 296 297 298 299 301 302 304 305 306 308 310 311 318 320 321 322 323
 324 325 326 328 329 332 335 336 337 338 339 340 343 344 347 350 351 353
 354 355 356 361 362 363 364 365 366 369 371 373 37

In [403]:
df = pd.read_pickle('/strombolihome/fribeiro/Thesis_project/Results/motif_enumeration_null_model/subgraph_3/subj01/fmri/motifs_size_3_fmri_0.11_time_point_13_dsk.pkl')
print(df)
print(df.iloc[0]['Frequency'])

            Subgraph Code Frequency Z-score Random_av Random_dev
011#100#100             1       536  -52.54   1555.13       19.4
011#101#110             2       395   52.54     55.29       6.47
536.0


In [363]:
G_array = createArrayGraph(dir_eeg_desikan + 'subj01-7T/conn_desi_cohi_time_eeg_alpha_correct_subj1-7T_.mat', 'eeg_alpha')
motifEnumerationRewiring(G_array, 0.07, 3, 1, 'eeg_alpha', 'dsk', '3')

[  0   1   2   3   4   5   6   8   9  11  13  15  17  19  20  21  22  26
  28  31  32  34  35  36  37  38  39  40  45  46  47  48  49  50  51  52
  54  55  56  57  58  59  61  62  63  64  65  66  67  68  69  70  71  72
  73  74  76  78  79  80  83  84  86  87  88  89  90  92  93  94  95  98
  99 100 102 103 104 105 106 107 109 111 113 118 119 120 122 124 126 131
 132 134 135 136 140 142 143 144 146 147 149 150 151 152 153 155 156 157
 159 160 164 165 167 168 169 170 174 176 177 178 179 180 181 182 183 184
 187 188 189 190 192 193 194 196 197 200 202 204 206 207 208 209 210 212
 213 214 216 218 219 220 221 223 224 225 227 229 230 231 235 237 239 240
 241 242 245 246 247 248 250 251 253 255 257 258 263 264 265 267 268 269
 270 272 274 275 276 277 279 280 281 282 283 285 286 287 291 292 293 294
 295 296 297 298 299 301 302 304 305 306 308 310 311 318 320 321 322 323
 324 325 326 328 329 332 335 336 337 338 339 340 343 344 347 350 351 353
 354 355 356 361 362 363 364 365 366 369 371 373 37

In [364]:
df = pd.read_pickle('/strombolihome/fribeiro/Thesis_project/Results/motif_enumeration_null_model/subgraph_3/subj01/eeg_alpha/motifs_size_3_eeg_alpha_0.07_time_point_13_dsk.pkl')
df

,Subgraph Code,Frequency,Z-score,Random_av,Random_dev
011#100#100,1,688,-10.61,878.86,17.98
011#101#110,2,107,10.61,43.38,5.99


### comparison with spatial null model 

In [5]:
from spaceCorrectedLouvainDC.Tools.spatialNullModel import *
from spaceCorrectedLouvainDC.Tools.utilSpatialNullM import *

#6. Function to compute distance between every pair of nodes - using Euclidean Distance
def getDistanceNodes(atlas):
    
    #coordinates = np.load('/strombolihome/fribeiro/Thesis_project/Results/coordinates_nodes_' + atlas + '.npy')
    if atlas == 'dsk':
        coordinates = np.loadtxt('/strombolihome/fribeiro/Dataset/desi_coord_68.txt')
    else:
        coordinates = np.loadtxt('/strombolihome/fribeiro/Dataset/destr_coords_simple_nosubc.txt')
    
    number_nodes = len(coordinates)
    f= open("/strombolihome/fribeiro/Thesis_project/Results/distance_between_nodes_" + atlas + ".txt",'a+')

    for i in range(0,number_nodes):
        for j in range(number_nodes):
            distance = np.linalg.norm(coordinates[i]-coordinates[j])
            f.write(str(i) + '_' + str(j) + '\t' + str(round(distance,4)) + ' ' +'\n')
            #print(str(i+1) + '_' + str(j+1) + ' ' + str(distance))
            
    f.close()

#7. Definition of class myDistances to have a map between each pair of nodes and distance between them, reading the .txt file
# (adapted from spaceCorrectedLouvainDC toolbox)
class myDistances():
    def __init__(self):
        self.allDistances = {}

    def getDistanceBetween(self,node1,node2):
        if not (str(node1),str(node2)) in self.allDistances:
            raise Exception(" distance from %s to %s unknown" %(node1,node2))
        return self.allDistances[(str(node1),str(node2))]

    def getDistanceFunctionVelov(self,file):
        f = open(file)
        for l in f:
            if l[0] != "#":
                elts = l.split("\t")
                (n1, n2) = elts[0].split("_")
                self.allDistances[(n1,n2)]=float(elts[1])
        return self.allDistances
    
#8. Definition of class to compute a deterrence function. First call deterrenceFunctionEstimation and then getDeterrenceAtDistance
# (adapted from spaceCorrectedLouvainDC toolbox)
class myDeterrenceFunction():
    def __init__(self):
        self.distancesDic = {}
        self.roundDecimals=-1
        self.maximalDistance=-1

    #changed so as to ignore the minVals parameter, as we have a low number of distance observations kept after
    #the graph's thresholding
    def deterrenceFunctionEstimation(self,INs,OUTs,observedGraph, distances, roundDecimals, minVals = 3, maximalDistance=100000, plot=False ):
        """
        Compute the deterrence function
        :param INs: dictionary of in-degrees
        :param OUTs: dictionary of out-degrees
        :param observedGraph: a nx.Graph , the observed network
        :param distances: a function that return the distance betwee two provided nodes
            :param roundDecimal: the rounding used to compute bins of the deterrence function. for a distance d=123.456 :
         if roundDecimal=2, binned value is 123.46.
         if roundDecimal=-2, binned value is 100
        :param maximalDistance: ignore in most cases, parameter of the deterence function to set an upper bound on the considered distances
        :param minValsBin: parameter of the deterrence function, minimum number of observations in a bin to consider it. (avoid abherent values for rare distances)
        :param plot: if True, plot the deterrence function before the doubly constrained process and at the end of the process
        """
        self.roundDecimals=roundDecimals
        self.maximalDistance=maximalDistance
        sumIn = sum(dict(observedGraph.in_degree(weight="weight")).values())

        byDistEstimated ={}
        byDistObserved ={}

        for e in observedGraph.edges(data=True):
            source = e[0]
            dest = e[1]


            theDist = distances(source,dest)
            #if theDist!=-1:
            theDist = _convertWithPrecision(theDist, roundDecimals)

            if theDist<maximalDistance:
                byDistEstimated.setdefault(theDist,[])
                byDistObserved.setdefault(theDist,[])

                byDistEstimated[theDist].append(OUTs[source]*INs[dest]/sumIn)
                byDistObserved[theDist].append(e[2]["weight"])

        dicCoeffdistance = {}
        for d in byDistObserved:
            #if len(byDistObserved[d])>minVals and sum(byDistEstimated[d])>0: #consider only if we have at least minVals values
            if sum(byDistEstimated[d])>0:
                dicCoeffdistance[d]=sum(byDistObserved[d])/sum(byDistEstimated[d])

        if plot:
            (x, y) = _fromDictionaryOutputOrderedKeysAndValuesByKey(dicCoeffdistance)
            _plotScatterFree([(x[:maximalDistance], y[:maximalDistance], "deterrence function")])
            
        self.distancesDic = dicCoeffdistance

    def getDeterrenceAtDistance(self,dist):
        """
        for a provided distance, return the associated deterrence
        :param dist: a distance
        """
        distNorm = _convertWithPrecision(dist, self.roundDecimals)
        if distNorm>=self.maximalDistance:
            return 0
        elif not distNorm in self.distancesDic:
            #return min(self.distancesDic.values())/10
            return 0
        return self.distancesDic[distNorm]

#9. Functions to estimate the intrinsic strength, correcting the in(out)-degree value with the deterrence function
# (adapted from spaceCorrectedLouvainDC toolbox - so as to avoid division by zero)
def _estimateEISsIN(EISsIN, EISsOUT, INs, OUTs, deterrencefunc, distances, normalized=False):
    #EIS : estimated Intrinsic Strenght
    newEISsIN = {}
    sumIn = sum(INs.values())

    #for each node
    for nodeDest in INs:
        #compute how many interaction it receives
        sumReceived =0.0
        for nodeSource in EISsOUT:
            sumReceived+=deterrencefunc(distances(nodeSource,nodeDest))*EISsOUT[nodeSource]*INs[nodeDest]/sumIn

        #print("Sum received:", sumReceived)
        #print("INs:", INs[nodeDest])
        #modify its "intrinsic degree" to receive the right number according to reference
        if sumReceived != 0:
            newEISsIN[nodeDest] = INs[nodeDest] / sumReceived * INs[nodeDest]
            #print("newEIS: ", newEISsIN[nodeDest])
        else:
            newEISsIN[nodeDest] = 0
        #print(nodeDest,sumReceived,INs[nodeDest],newEISsIN[nodeDest])

    return newEISsIN

def _estimateEISsOUT(EISsIN, EISsOUT, INs, OUTs, deterrencefunc, distances, normalized=False):
    # EIS : estimated Intrinsic Strenght
    newEISsOUT = {}
    sumOut = sum(OUTs.values())

    # for each node
    for nodeSource in OUTs:
        # compute how many interaction it receives
        sumSent = 0.0
        for nodeDest in EISsIN:
            sumSent += deterrencefunc(distances(nodeSource, nodeDest)) * EISsIN[nodeDest] * OUTs[nodeSource]/sumOut
        
        #print("Sum sent:", sumSent)
        #print("OUTs:", OUTs[nodeSource])
        if sumSent != 0:
            # modify its "intrinsic degree" to receive the right number according to reference
            newEISsOUT[nodeSource] = OUTs[nodeSource] / sumSent * OUTs[nodeSource]
        else:
            newEISsOUT[nodeSource] = 0

    return newEISsOUT

def _convertWithPrecision(val, precision):
    #precision of rounding as a distance to the "." for instance :
    # 123.456 with precision 2 = 123.46
    # 123.456 with precision -2 = 100
    if precision >= 0:
        theDist = round(val, precision)
    else:
        theDist = int(val / math.pow(10, abs(precision))) * math.pow(10, abs(
            precision))  # if negative, round to closest dimension
    return theDist

In [31]:
#10. Function to generate spatial null model, gravity-based but including constraints regarding the degree distribution
# (adapted from spaceCorrectedLouvainDC toolbox - so as to use the adapted version of the deterrence function class)
def getSpatialNullModelAdapted(originalNetwork,distances,roundDecimal,maximalDistance=100000,minValsBin = 3,plot=False,iterations=5,printDebug=False):
    """
    :param originalNetwork: the observed network for which we want the corresponding null model
    :param distances: a function that return the distance between two nodes of the provided graph
    :param roundDecimal: the rounding used to compute bins of the deterrence function. for a distance d=123.456 :
     if roundDecimal=2, binned value is 123.46.
     if roundDecimal=-2, binned value is 100
    :param maximalDistance: ignore in most cases, parameter of the deterence function to set an upper bound on the considered distances
    :param minValsBin: parameter of the deterrence function, minimum number of observations in a bin to consider it. (avoid abherent values for rare distances)
    :param plot: if True, plot the deterrence function before the doubly constrained process and at the end of the process
    :param iterations: number of iterations in the doubly constrained process
    :param printDebug: print at each step a trace of the current model: edit distance with original network, degree bias towards central nodes...
    slow down the process a lot, use only to understand or check that everything is going well.
    """
    print("Computing the spatial null model with %s iterations"%(iterations))
    norm = False
    GraphModelOriginal = GraphModelAsnxGraph(originalNetwork)

    # compute in and out degrees
    INs = dict(originalNetwork.in_degree(weight="weight"))
    OUTs = dict(originalNetwork.out_degree(weight="weight"))


    print("computing the original deterrence function")
    deterrencefunc = myDeterrenceFunction()
    deterrencefunc.deterrenceFunctionEstimation(INs, OUTs, originalNetwork, distances,
                                                      roundDecimals=roundDecimal, maximalDistance=maximalDistance, plot=plot,minVals = minValsBin)

    EISsIN=INs
    EISsOUT=OUTs

    for i in range(iterations):
        print("----------step %s" %(i))

        # as we wish to have the same estimated intrinsic strength both in and out - undirected graph with same weight
        EISsIN = _estimateEISsIN(EISsIN, EISsOUT, INs, OUTs, deterrencefunc.getDeterrenceAtDistance, distances, normalized=norm)
        #EISsOUT = _estimateEISsOUT(EISsIN, EISsOUT, INs, OUTs, deterrencefunc.getDeterrenceAtDistance, distances, normalized=norm)
        EISsOUT = EISsIN

        
        deterrencefunc = myDeterrenceFunction()
        deterrencefunc.deterrenceFunctionEstimation(EISsIN, EISsOUT, originalNetwork, distances,
                                                    roundDecimals=roundDecimal, maximalDistance=maximalDistance,
                                                    plot=False,minVals = minValsBin)

    if plot:
        deterrencefunc = myDeterrenceFunction()
        deterrencefunc.deterrenceFunctionEstimation(EISsIN, EISsOUT, originalNetwork, distances,
                                                    roundDecimals=roundDecimal, maximalDistance=maximalDistance,
                                                    plot=plot,minVals = minValsBin)


    temporaryModel = GravityModel(EISsIN, EISsOUT, deterrencefunc.getDeterrenceAtDistance, distances,
                                  desiredInDegrees=INs, desiredOutDegrees=OUTs)


    return(temporaryModel,deterrencefunc)

def myCreatenxGraphFromGraphModel(graphModel):
    """
    :param graphModel:
    :return:
    """
    aGraph = nx.Graph()
    for source in graphModel.getNodes():
        for dest in graphModel.getNodes():
            aGraph.add_edge(source, dest, weight=0)

    for source in graphModel.getNodes():
        for dest in graphModel.getNodes():
            if aGraph[source][dest]["weight"] == 0:
                aGraph[source][dest]["weight"] += graphModel.getExpectedEdges(source,dest)
            else: 
                aGraph[source][dest]["weight"] = (aGraph[source][dest]["weight"] + graphModel.getExpectedEdges(source,dest))/2
    
    
    return aGraph   

In [32]:
import random

#11. Function to generate ensemble of surrogates from spatial null model
def ensembleSpatialNullModel(conn_matrix, number_networks, atlas):
    
    if atlas == 'dsk':
        num_regions = 68
    else:
        num_regions = 148
        
    #conversion to binary directed graph to use the spatialNullModel function
    G = nx.from_numpy_matrix(conn_matrix)
    G = nx.to_directed(G)
    
    #get the distance between all pairs of nodes from txt file
    distanceFile = "/strombolihome/fribeiro/Thesis_project/Results/distance_between_nodes_" + atlas + ".txt"
    distancesBetweenNodes = myDistances()
    all_distances = distancesBetweenNodes.getDistanceFunctionVelov(distanceFile)
    distances = distancesBetweenNodes.getDistanceBetween
    
    ensemble_null_model = np.empty(number_networks, dtype=object)
    
    #compute the null model of a spatial network
    (nullModel, deterrenceFunction) = getSpatialNullModelAdapted(G,distances,roundDecimal=3,maximalDistance=100000,minValsBin = 3,plot=False,iterations=5,printDebug=False)
    
    #compute undirected version of null model - obtain edge probability matrix
    graph_model = myCreatenxGraphFromGraphModel(nullModel)
    prob_matrix = nx.to_numpy_matrix(graph_model)
    
    ind_keep = np.transpose(np.nonzero(prob_matrix))
    idx_i, idx_j = zip(*ind_keep)
    
    for n in range(0,number_networks):
        
        conn_matrix = np.zeros((num_regions,num_regions))
        random_value = np.array([[random.uniform(0,1) for i in range(num_regions)] for j in range(num_regions)])
        
        # positive values correspond to random < probability and negative values to random > probability
        conn_matrix[idx_i, idx_j] = prob_matrix[idx_i,idx_j]-random_value[idx_i,idx_j]
        
        # so as to remove edges for which random value generated is bigger than existing probability
        ind_remove = np.argwhere(conn_matrix<0)
        if len(ind_remove) != 0:
            idx_u, idx_v = zip(*ind_remove)
            conn_matrix[idx_u,idx_v] = 0
        
        #print(conn_matrix)
        
        ensemble_null_model[n] = bct.utils.binarize(conn_matrix)
            
    return ensemble_null_model
    

In [37]:
# to add frequency values according to spatial null model
def arrangeMotifResultsSpatial(file, frequency_values, s, mode):
    
    f = open("/strombolihome/fribeiro/Thesis_project/Code/results.txt", 'r')
    lines = f.readlines() 
    words = []
    i = 1
    for line in lines: 
        line = format(line.strip())
        # when the results start
        if i >= 29:
            #to remove additional characters
            out = re.split(' |, |\*|\n',line)
            out = [o for o in out if o != '|'] 
            out = [o for o in out if o != '' ]
            out = [o for o in out if o != '+/-']
            if len(out) > 0:
                words.append(out[:])
        i += 1  
    f.close()
    
    #if studying motifs with size 3 - 2 types 
    if mode == '3':
        code = ''
        for j in range(0,len(words)):
            j += 1
            #print("make codes")
            if j%3 != 0:
                code += str(words[j-1][0]) + "#"
            elif j%3 == 0:
                code += str(words[j-1][0])
            if code == '011#100#100':
                frequency_values[0,s] = words[j-1][1] 
                code = ''
            elif code == '011#101#110':
                frequency_values[1,s] = words[j-1][1] 
                code = ''
        
        return frequency_values
    
    #or if studying motifs with size 4 - 6 types     
    elif mode == '4':
        code = ''
        for j in range(0,len(words)):
            j += 1
            if j%4 != 0:
                code += str(words[j-1][0]) + "#"
            elif j%4 == 0:
                code += str(words[j-1][0])
            if code == '0110#1001#1000#0100':
                frequency_values[0,s] = words[j-1][1] 
                code = ''
            elif code == '0111#1010#1100#1000':
                frequency_values[1,s] = words[j-1][1] 
                code = ''
            elif code == '0111#1000#1000#1000':
                frequency_values[2,s] = words[j-1][1] 
                code = ''
            elif code == '0111#1011#1100#1100':
                frequency_values[3,s] = words[j-1][1] 
                code = ''
            elif code == '0111#1011#1101#1110':
                frequency_values[4,s] = words[j-1][1] 
                code = ''
            elif code == '0110#1001#1001#0110':
                frequency_values[5,s] = words[j-1][1] 
                code = ''
        
        return frequency_values
    
    #or if studying motifs with size 5 - 21 types             
    elif mode == '5':
        code = ''
        for j in range(0,len(words)):
            j += 1
            if j%5 != 0:
                code += str(words[j-1][0]) + "#"
            elif j%5 == 0:
                code += str(words[j-1][0])
            if code == '01100#10010#10001#01000#00100':
                frequency_values[0,s] = words[j-1][1] 
                code = ''
            elif code == '01110#10001#10000#10000#01000':
                frequency_values[1,s] = words[j-1][1] 
                code = ''
            elif code == '01110#10100#11000#10001#00010':
                frequency_values[2,s] = words[j-1][1] 
                code = ''
            elif code == '01110#10101#11000#10000#01000':
                frequency_values[3,s] = words[j-1][1] 
                code = ''
            elif code == '01110#10110#11001#11000#00100':
                frequency_values[4,s] = words[j-1][1] 
                code = ''
            elif code == '01111#10100#11000#10000#10000':
                frequency_values[5,s] = words[j-1][1] 
                code = ''
            elif code == '01111#10110#11000#11000#10000':
                frequency_values[6,s] = words[j-1][1] 
                code = ''
            elif code == '01111#10110#11010#11100#10000':
                frequency_values[7,s] = words[j-1][1] 
                code = ''
            elif code == '01111#10110#11001#11000#10100':
                frequency_values[8,s] = words[j-1][1] 
                code = ''
            elif code == '01100#10011#10010#01100#01000':
                frequency_values[9,s] = words[j-1][1] 
                code = ''
            elif code == '01111#10111#11010#11100#11000':
                frequency_values[10,s] = words[j-1][1] 
                code = ''
            elif code == '01111#10100#11000#10001#10010':
                frequency_values[11,s] = words[j-1][1] 
                code = ''
            if code == '01111#10000#10000#10000#10000':
                frequency_values[12,s] = words[j-1][1] 
                code = ''
            elif code == '01101#10011#10010#01100#11000':
                frequency_values[13,s] = words[j-1][1] 
                code = ''
            elif code == '01111#10111#11011#11100#11100':
                frequency_values[14,s] = words[j-1][1] 
                code = ''
            elif code == '01111#10111#11000#11000#11000':
                frequency_values[15,s] = words[j-1][1] 
                code = ''
            elif code == '01100#10010#10001#01001#00110':
                frequency_values[16,s] = words[j-1][1] 
                code = ''
            elif code == '01110#10110#11001#11001#00110':
                frequency_values[17,s] = words[j-1][1] 
                code = ''
            elif code == '01111#10110#11001#11001#10110':
                frequency_values[18,s] = words[j-1][1] 
                code = ''
            elif code == '01111#10111#11011#11101#11110':
                frequency_values[19,s] = words[j-1][1] 
                code = ''
            elif code == '01100#10011#10011#01100#01100':
                frequency_values[20,s] = words[j-1][1] 
                code = ''
                
        return frequency_values

In [44]:
import shlex, subprocess
import re
import pandas as pd
import time

#function to get all types of subgraphs, their frequency and z-score compared to a spatial null model (100 networks)
# done with significant time points common between fMRI and EEG alpha
def motifEnumerationSpatial(G_array, threshold, delay, subject, type_data, atlas, mode):
    
    #set args for running c++ code and to store results
    if mode == '3':
        command_line = './gtrieScanner_src_01/gtrieScanner -s 3 -m gtrie ./gtrieScanner_src_01/gtries/undir3.gt -g graph.txt ' \
                        + '-u'
    elif mode == '4':
        command_line = './gtrieScanner_src_01/gtrieScanner -s 4 -m gtrie ./gtrieScanner_src_01/gtries/undir4.gt -g graph.txt ' \
                        + '-u'
    elif mode == '5':
        command_line = './gtrieScanner_src_01/gtrieScanner -s 5 -m gtrie ./gtrieScanner_src_01/gtries/undir5.gt -g graph.txt ' \
                        + '-u'
        
    significant_time_points_spatial = np.load('/strombolihome/fribeiro/Thesis_project/Results/comparison_spatial_null_model/significant_time_points/subj0' + str(subject) + '/significant_time_points_fmri_delay_' + str(delay) + '_eeg_alpha_' + atlas + '.npy')
    significant_time_points = significant_time_points_spatial.astype(int)
    significant_time_points = significant_time_points - 1

    for t in significant_time_points:
        print(t)
        
        if type_data == 'fmri':
            G, conn_matrix, min_pos, max_neg = thresholdGraph(G_array[t+delay], threshold, 'fmri', 'percentage')
        else:
            G, conn_matrix, min_coh = thresholdGraph(G_array[t], threshold, type_data, 'percentage')
        
        #conversion to binary undirected graph
        conn_matrix = bct.utils.binarize(conn_matrix)
        G_original = nx.from_numpy_matrix(conn_matrix)

        f = open("/strombolihome/fribeiro/Thesis_project/Code/graph.txt", "a")
        f.truncate(0)
        for edge in list(G_original.edges(data=True)):
            f.write(str(edge[1]+1) + ' ' + str(edge[0]+1) + ' ' + str(int(G_original[edge[0]][edge[1]]["weight"])) + '\n')
        f.close()

        args = shlex.split(command_line)
        p = subprocess.Popen(args)

        time.sleep(0.5)
        
        # store results in a dataframe for easy access and to clear out extra information
        df = arrangeMotifResults('/strombolihome/fribeiro/Thesis_project/Code/results.txt', mode)

        ensemble_spatial_null_model = ensembleSpatialNullModel(conn_matrix, 100, 'dsk')
        
        #if studying motifs with size 3 - 2 types 
        if mode == '3':
            num_types = 2
        #or if studying motifs with size 4 - 6 types
        elif mode == '4':
            num_types = 6
        #or if studying motifs with size 5 - 21 types
        elif mode == '5':
            num_types = 21
        frequency_values = np.zeros((num_types,100))
            
        for s in range(100):
    
            G_null = nx.from_numpy_matrix(ensemble_spatial_null_model[s])

            # to reset and write in graph representation txt
            f = open("/strombolihome/fribeiro/Thesis_project/Code/graph.txt", "a")
            f.truncate(0)
            for edge in list(G_null.edges(data=True)):
                f.write(str(edge[1]+1) + ' ' + str(edge[0]+1) + ' ' + str(int(G_null[edge[0]][edge[1]]["weight"])) + '\n')
            f.close()

            args = shlex.split(command_line)
            p = subprocess.Popen(args)

            time.sleep(0.5)
            
            frequency_values = arrangeMotifResultsSpatial('/strombolihome/fribeiro/Thesis_project/Code/results.txt', frequency_values, s, mode)
           
        #add z-score, average frequency and standard deviation for spatial null model
        for n in range(0,num_types):
            df.iloc[n]['Random_av'] = round(np.mean(frequency_values[n,:]),2)
            df.iloc[n]['Random_dev'] = round(np.std(frequency_values[n,:]),2)
            df.iloc[n]['Z-score'] = round((float(df.iloc[n]['Frequency'])- df.iloc[n]['Random_av'])/df.iloc[n]['Random_dev'],2)
        
        print(df)
        df.to_pickle('/strombolihome/fribeiro/Thesis_project/Results/motif_enumeration_spatial_null_model/subgraph_' + mode \
                        + '/subj0' + str(subject) + '/' + type_data + '/motifs_size_' + mode + '_' + type_data + '_' + str(threshold) + '_time_point_' + str(t) + '_' + atlas + '.pkl')
        break

In [45]:
G_array = createArrayGraph(dir_fmri_desikan + 'subj01-7T/conn_desi_phase_coh_time_fmri.mat', 'fmri')
motifEnumerationSpatial(G_array, 0.11, 3, 1, 'fmri', 'dsk', '3')

9
Computing the spatial null model with 5 iterations
computing the original deterrence function
----------step 0
----------step 1
----------step 2
----------step 3
----------step 4
            Subgraph Code Frequency Z-score Random_av Random_dev
011#100#100             1       487   -8.36    791.52      36.43
011#101#110             2       444   -0.36    446.06       5.75


In [30]:
G_array = createArrayGraph(dir_fmri_desikan + 'subj01-7T/conn_desi_phase_coh_time_fmri.mat', 'fmri')

subject = 1
atlas = 'dsk'
delay = 3

#significant_time_points_spatial = np.load('/strombolihome/fribeiro/Thesis_project/Results/comparison_spatial_null_model/significant_time_points/subj0' + str(subject) + '/significant_time_points_fmri_delay_' + str(delay) + '_eeg_alpha_' + atlas + '.npy')
#significant_time_points = significant_time_points_spatial.astype(int)
#significant_time_points = significant_time_points - 1

#for t in significant_time_points:
        #print(t)
    
G, conn_matrix, min_pos, max_neg = thresholdGraph(G_array[9+delay], 0.11, 'fmri', 'percentage')
conn_matrix = bct.utils.binarize(conn_matrix)

ensemble_spatial_null_model = ensembleSpatialNullModel(conn_matrix, 100, 'dsk')

command_line = './gtrieScanner_src_01/gtrieScanner -s 3 -m gtrie ./gtrieScanner_src_01/gtries/undir3.gt -g graph.txt ' \
                        + '-u'

frequency_values = np.zeros((2,100))

for s in range(100):
    
    G_null = nx.from_numpy_matrix(ensemble_spatial_null_model[s])

    # to reset and write in graph representation txt
    f = open("/strombolihome/fribeiro/Thesis_project/Code/graph.txt", "a")
    f.truncate(0)
    for edge in list(G_null.edges(data=True)):
        f.write(str(edge[1]+1) + ' ' + str(edge[0]+1) + ' ' + str(int(G_null[edge[0]][edge[1]]["weight"])) + '\n')
    f.close()

    args = shlex.split(command_line)
    p = subprocess.Popen(args)

    time.sleep(0.5)

    f = open("/strombolihome/fribeiro/Thesis_project/Code/results.txt", 'r')
    lines = f.readlines() 
    words = []
    i = 1
    for line in lines: 
        line = format(line.strip())
        #print(line)
        # when the results start
        if i >= 29:
            #to remove additional characters
            out = re.split(' |, |\*|\n',line)
            out = [o for o in out if o != '|'] 
            out = [o for o in out if o != '' ]
            out = [o for o in out if o != '+/-']
            if len(out) > 0:
                words.append(out[:])

        i += 1

    print(words)
    f.close()
    
    code = ''
    for j in range(0,len(words)):
        j += 1
        #print("make codes")
        if j%3 != 0:
            code += str(words[j-1][0]) + "#"
        elif j%3 == 0:
            code += str(words[j-1][0])
        print(code)
        if code == '011#100#100':
            print(words[j-1][1])
            frequency_values[0,s] = words[j-1][1] 
            code = ''
        elif code == '011#101#110':
            print(words[j-1][1])
            frequency_values[1,s] = words[j-1][1] 
            code = ''

av_frequency_code_1 = round(np.mean(frequency_values[0,:]),2)
print(av_frequency_code_1)
av_frequency_code_2 = round(np.mean(frequency_values[1,:]),2)
print(av_frequency_code_2)

dev_frequency_code_1 = round(np.std(frequency_values[0,:]),2)
print(dev_frequency_code_1)
dev_frequency_code_2 = round(np.std(frequency_values[1,:]),2)
print(dev_frequency_code_2)

Computing the spatial null model with 5 iterations
computing the original deterrence function
----------step 0
----------step 1
----------step 2
----------step 3
----------step 4
None


TypeError: 'NoneType' object is not iterable

In [415]:
G_original = nx.from_numpy_matrix(conn_matrix)

f = open("/strombolihome/fribeiro/Thesis_project/Code/graph.txt", "a")
f.truncate(0)
for edge in list(G_original.edges(data=True)):
    f.write(str(edge[1]+1) + ' ' + str(edge[0]+1) + ' ' + str(int(G_original[edge[0]][edge[1]]["weight"])) + '\n')
f.close()

args = shlex.split(command_line)
p = subprocess.Popen(args)

time.sleep(0.5)

f = open("/strombolihome/fribeiro/Thesis_project/Code/results.txt", 'r')
lines = f.readlines() 
words = []
i = 1
for line in lines: 
    line = format(line.strip())
    #print(line)
    # when the results start
    if i >= 29:
        #to remove additional characters
        out = re.split(' |, |\*|\n',line)
        out = [o for o in out if o != '|'] 
        out = [o for o in out if o != '' ]
        out = [o for o in out if o != '+/-']
        if len(out) > 0:
            words.append(out[:])

    i += 1

print(words)
f.close()

df = pd.DataFrame(columns = ['Subgraph Code', 'Frequency', 'Z-score', 'Random_av', 'Random_dev'], 
                              index = ['011#100#100', '011#101#110'])
code = ''
for j in range(0,len(words)):
    j += 1
    #print("make codes")
    if j%3 != 0:
        code += str(words[j-1][0]) + "#"
    elif j%3 == 0:
        code += str(words[j-1][0])
    #print(code)
    if code == '011#100#100':
        #print("here")
        df.loc[code] = ['1', float(words[j-1][1]), float(words[j-1][2]), float(words[j-1][3]), float(words[j-1][4])]  
        code = ''
    elif code == '011#101#110':
        #print("here 2")
        df.loc[code] = ['2', float(words[j-1][1]), float(words[j-1][2]), float(words[j-1][3]), float(words[j-1][4])]  
        code = ''
df

[['011'], ['100'], ['100', '487', '0.00', '0.00', '0.00'], ['011'], ['101'], ['110', '444', '0.00', '0.00', '0.00']]


,Subgraph Code,Frequency,Z-score,Random_av,Random_dev
011#100#100,1,487,0,0,0
011#101#110,2,444,0,0,0


In [411]:
z_score_1 = round((float(df.iloc[0]['Frequency'])- round(av_frequency_code_1,2))/round(dev_frequency_code_1,2),2)
print(z_score_1)
z_score_2 = round((float(df.iloc[1]['Frequency'])- round(av_frequency_code_2,2))/round(dev_frequency_code_2,2),2)
print(z_score_2)

-7.75
-0.19


In [416]:
z_score = [z_score_1, z_score_2] 
av_frequency = [av_frequency_code_1, av_frequency_code_2]
dev_frequency = [dev_frequency_code_1, dev_frequency_code_2]
df.iloc[0]['Z-score'] = z_score_1
df.iloc[1]['Z-score'] = z_score_2


In [417]:
df

,Subgraph Code,Frequency,Z-score,Random_av,Random_dev
011#100#100,1,487,-7.75,0,0
011#101#110,2,444,-0.19,0,0


In [418]:
df.iloc[0]['Z-score']

-7.75